In [4]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import (Input, InputLayer, Conv2D, MaxPool2D,
                                     Dense, Flatten, BatchNormalization)
from tensorflow.keras import Model, Sequential
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy

print(tf.__version__)

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/co

2.20.0


In [5]:
import tensorflow_datasets as tfds
print(tfds.__file__)
print(tfds.__version__ if hasattr(tfds, "__version__") else "no version attr")
print([a for a in dir(tfds) if not a.startswith("_")][:20])

/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py
no version attr
['annotations', 'audio', 'd4rl', 'datasets', 'graphs', 'image', 'image_classification', 'logging', 'nearest_neighbors', 'object_detection', 'question_answering', 'ranking', 'recommendation', 'rl_unplugged', 'robomimic', 'robotics', 'structured', 'summarization', 'text', 'text_simplification']


In [6]:
import os, glob

#
print("cwd:", os.getcwd())
print([f for f in os.listdir('.') if 'tensorflow' in f or 'tfds' in f])

#
PROJECT = '/content/drive/MyDrive/malaria_cnn_project'
print(glob.glob(f"{PROJECT}/**/tensorflow_datasets*", recursive=True))
print(glob.glob(f"{PROJECT}/**/tfds.py", recursive=True))

cwd: /content
[]
[]
[]


In [7]:
!pip install -q --upgrade "protobuf>=6.31.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.


In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
from google.protobuf import __version__ as pb
print("TF:", tf.__version__, "| protobuf:", pb, "| tfds:", tfds.__version__)
print("has load:", hasattr(tfds, "load"))   # True আসতে হবে

TF: 2.20.0 | protobuf: 5.29.6 | tfds: 4.9.10
has load: True


In [3]:
!pip install -q "protobuf==5.29.6" "tensorflow_metadata<1.17"

In [4]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/malaria_cnn_project/tfds_data'

dataset, dataset_info = tfds.load(
    'malaria', with_info=True, as_supervised=True,
    shuffle_files=True, split=['train'], data_dir=DATA_DIR
)
print(dataset_info.splits['train'].num_examples)   # 27558

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
27558


# setup

In [5]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import (Input, InputLayer, Conv2D, MaxPool2D,
                                     Dense, Flatten, BatchNormalization, Layer)
from tensorflow.keras import Model, Sequential
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam

CONFIG = {
    "im_size": 224,
    "batch_size": 32,
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "learning_rate": 0.01,
    "epochs": 3,
    "data_dir": "/content/drive/MyDrive/malaria_cnn_project/tfds_data",
}

tf.random.set_seed(42)
print(tf.__version__)

2.20.0


# Data pipeline

In [6]:
dataset, dataset_info = tfds.load(
    'malaria', with_info=True, as_supervised=True,
    shuffle_files=True, split=['train'], data_dir=CONFIG["data_dir"]
)

def splits(dataset, train_ratio, val_ratio):
    size = len(dataset)
    train = dataset.take(int(train_ratio * size))
    rest  = dataset.skip(int(train_ratio * size))
    val   = rest.take(int(val_ratio * size))
    test  = rest.skip(int(val_ratio * size))
    return train, val, test

def resize_rescale(image, label, im_size):
    return tf.image.resize(image, (im_size, im_size)) / 255.0, label

def prepare(ds, im_size, batch_size, shuffle=True):
    ds = ds.map(lambda x, y: resize_rescale(x, y, im_size),
                num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=8, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_raw, val_raw, test_raw = splits(dataset[0], CONFIG["train_ratio"], CONFIG["val_ratio"])

train_ds = prepare(train_raw, CONFIG["im_size"], CONFIG["batch_size"])
val_ds   = prepare(val_raw,   CONFIG["im_size"], CONFIG["batch_size"])
test_ds  = prepare(test_raw,  CONFIG["im_size"], 1, shuffle=False)

print(len(train_raw), len(val_raw), len(test_raw))

22046 2755 2757


# Sequential

In [7]:
def build_lenet_sequential(im_size=224):
    return Sequential([
        InputLayer(input_shape=(im_size, im_size, 3)),
        Conv2D(6, 3, 1, padding='valid', activation='relu'),
        BatchNormalization(),
        MaxPool2D(2, 2),
        Conv2D(16, 3, 1, padding='valid', activation='relu'),
        BatchNormalization(),
        MaxPool2D(2, 2),
        Flatten(),
        Dense(100, activation='relu'),
        BatchNormalization(),
        Dense(10, activation='relu'),
        BatchNormalization(),
        Dense(1, activation='sigmoid'),
    ], name="lenet_sequential")


m = build_lenet_sequential(CONFIG["im_size"])
m.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "lenet_sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 6)    │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 6)    │            24 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 16)   │           880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 46656)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │     4,665,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 10)             │            40 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,668,297 (17.81 MB)

 Trainable params: 4,668,033 (17.81 MB)

 Non-trainable params: 264 (1.03 KB)

# same mode but with  Functional API

In [8]:
def build_lenet_functional(im_size=224):
    func_input = Input(shape=(im_size, im_size, 3), name="input_image")

    x = Conv2D(6, 3, 1, padding='valid', activation='relu')(func_input)
    x = BatchNormalization()(x)
    x = MaxPool2D(2, 2)(x)

    x = Conv2D(16, 3, 1, padding='valid', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPool2D(2, 2)(x)

    x = Flatten()(x)

    x = Dense(100, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dense(10, activation='relu')(x)
    x = BatchNormalization()(x)

    func_output = Dense(1, activation='sigmoid')(x)

    return Model(func_input, func_output, name="lenet_functional")


lenet_func = build_lenet_functional(CONFIG["im_size"])
lenet_func.summary()

Model: "lenet_functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 6)    │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 222, 222, 6)    │            24 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 111, 111, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 16)   │           880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 109, 109, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 54, 54, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 46656)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │     4,665,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         1,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 10)             │            40 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,668,297 (17.81 MB)

 Trainable params: 4,668,033 (17.81 MB)

 Non-trainable params: 264 (1.03 KB)

In [9]:
assert (build_lenet_sequential(224).count_params()
        == build_lenet_functional(224).count_params())
print("✅ একই মডেল, ভিন্ন লেখার ধরন")

✅ একই মডেল, ভিন্ন লেখার ধরন


In [10]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import Model

inputs  = Input(shape=(784,))
x       = Dense(64, activation='relu')(inputs)
outputs = Dense(10, activation='softmax')(x)

model = Model(inputs, outputs)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)

# Simple Functional model

In [11]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import (Input, InputLayer, Conv2D, MaxPool2D,
                                     Dense, Flatten, BatchNormalization, Layer)
from tensorflow.keras import Model, Sequential
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam

In [12]:
inputs=Input(shape=(784,))

x=Dense(64,activation='relu')(inputs)


outputs=Dense(10,activation='softmax')(x)


model=Model(inputs=inputs,outputs=outputs)


model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)

# Multiple Inputs

In [13]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

inputA = Input(shape=(64,), name="input_A") # 64 value/features for each sample
inputB = Input(shape=(128,), name="input_B")

# input a is 64-feature input tensor not model.

In [14]:
print(inputA)
print(inputB)

<KerasTensor shape=(None, 64), dtype=float32, sparse=False, ragged=False, name=input_A>
<KerasTensor shape=(None, 128), dtype=float32, sparse=False, ragged=False, name=input_B>


In [15]:
x = Dense(8, activation='relu', name="A_dense_1")(inputA)

x = Dense(4, activation='relu', name="A_dense_2")(x)

In [20]:
print(x)

<KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_64>


# process input a

In [16]:
a=Dense(8,activation='relu',name="a_dense_1")(inputA)

In [17]:
print(a)

<KerasTensor shape=(None, 8), dtype=float32, sparse=False, ragged=False, name=keras_tensor_58>


In [ ]:
# Dense layer of 8 neurons  ,a tensor of 8 values , inputA  tensor of 64 values

In [18]:
a=Dense(4,activation='relu',name="a_dense_2")(a)

In [19]:
Branch_a_model=Model(inputs=inputA,outputs=a,name="branch_a")


#inputA থেকে a পর্যন্ত পুরো connected path-টাকে একটা Model বানাও।


In [22]:
Branch_a_model.summary()

Model: "branch_a"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_A (InputLayer)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ a_dense_1 (Dense)               │ (None, 8)              │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ a_dense_2 (Dense)               │ (None, 4)              │            36 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 556 (2.17 KB)

 Trainable params: 556 (2.17 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
print("Input A:", inputA)
print("Output a:", a)

print("Model input :", Branch_a_model.input)
print("Model output:", Branch_a_model.output)

Input A: <KerasTensor shape=(None, 64), dtype=float32, sparse=False, ragged=False, name=input_A>
Output a: <KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_59>
Model input : <KerasTensor shape=(None, 64), dtype=float32, sparse=False, ragged=False, name=input_A>
Model output: <KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_59>


# Branch B

In [25]:
inputB = Input(shape=(128,), name="input_B")

In [26]:
b = Dense(
    16,
    activation='relu',
    name="b_dense_1"
)(inputB)

In [27]:
print(b)

<KerasTensor shape=(None, 16), dtype=float32, sparse=False, ragged=False, name=keras_tensor_60>


In [28]:
b = Dense(
    4,
    activation='relu',
    name="b_dense_2"
)(b)

In [29]:
print(b)

<KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_61>


In [30]:
Branch_b_model = Model(
    inputs=inputB,
    outputs=b,
    name="branch_b"
)

In [31]:
Branch_b_model.summary()

Model: "branch_b"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_B (InputLayer)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b_dense_1 (Dense)               │ (None, 16)             │         2,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b_dense_2 (Dense)               │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,132 (8.33 KB)

 Trainable params: 2,132 (8.33 KB)

 Non-trainable params: 0 (0.00 B)

# Concatenate

In [32]:
from tensorflow.keras.layers import Concatenate

In [33]:
combined = Concatenate()([
    Branch_a_model.output,
    Branch_b_model.output
])

In [34]:
print(combined)

<KerasTensor shape=(None, 8), dtype=float32, sparse=False, ragged=False, name=keras_tensor_62>


# Combined features process

In [35]:
z = Dense(
    2,
    activation='relu',
    name="combined_dense"
)(combined)

In [36]:
print(z)

<KerasTensor shape=(None, 2), dtype=float32, sparse=False, ragged=False, name=keras_tensor_63>


In [37]:
output = Dense(
    1,
    activation='linear',
    name="output"
)(z)

In [38]:
print(output)

<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_64>


In [39]:
multi_input_model = Model(
    inputs=[
        Branch_a_model.input,
        Branch_b_model.input
    ],
    outputs=output,
    name="multi_input_model"
)

In [40]:
multi_input_model.summary()

Model: "multi_input_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_A             │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_B             │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ a_dense_1 (Dense)   │ (None, 8)         │        520 │ input_A[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b_dense_1 (Dense)   │ (None, 16)        │      2,064 │ input_B[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ a_dense_2 (Dense)   │ (None, 4)         │         36 │ a_dense_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b_dense_2 (Dense)   │ (None, 4)         │         68 │ b_dense_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 8)         │          0 │ a_dense_2[0][0],  │
│ (Concatenate)       │                   │            │ b_dense_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 2)         │         18 │ concatenate[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │          3 │ combined_dense[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,709 (10.58 KB)

 Trainable params: 2,709 (10.58 KB)

 Non-trainable params: 0 (0.00 B)